# Part 5 · Noise & Realistic Execution

Everything so far ran on an **ideal** simulator: every gate perfect, every qubit immortal, every measurement honest. Real quantum hardware is none of those things. Qubits leak their state to the environment, gates over- and under-rotate, and readout sometimes reports the wrong bit. Noise is not a footnote: it is **the central obstacle to useful quantum computers**, and error correction and error mitigation are entire research fields because of it.

This part shows how QiliSDK models noise, and the punchline is delightfully simple: **going from ideal to noisy is a single argument on the backend**, `QiliSim(noise_model=...)`. The functional and the readout do not change at all. That makes a noise model your **staging environment**: you can test whether an algorithm survives realistic hardware before paying for QPU time (QPU = quantum processing unit, the actual quantum chip).

By the end of this notebook you will be able to:

- explain why noise forces us from state *vectors* to **density matrices**, and tell a superposition from a classical mixture;
- build a **`NoiseModel`** from physical noise channels and attach it to `QiliSim`;
- reproduce a real lab measurement, **$T_1$ relaxation**, using datasheet transmon numbers;
- answer a question with real engineering stakes: **how much gate noise can Part 4's chemistry result tolerate?**

On the `overall.jpg` architecture map, **Noise Models** sit in the Execution layer: they are applied *while* a program runs, orthogonal to the program you wrote.

In [ ]:
# ▶ Run me first. No-op if QiliSDK is already installed; installs it on a fresh env (e.g. Google Colab).
try:
    import qilisdk
except ImportError:
    import sys
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "qilisdk[openqasm,qir]==0.2.0", "matplotlib", "numpy"], check=True)
    import qilisdk  # Colab: if this still fails, Runtime > Restart session, then rerun
print("QiliSDK", qilisdk.__version__)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 5.0 · From pure states to density matrices

Why do qubits err at all? A qubit is a physical system (a superconducting circuit, a trapped ion, a photon), and it is never perfectly isolated. Its environment continuously and uncontrollably interacts with it, and that slow leak of quantum information is called **decoherence**. Three failure modes dominate every hardware datasheet:

- **Relaxation ($T_1$):** an excited qubit decays, $|1\rangle \to |0\rangle$, like a leaky capacitor losing its charge. $T_1$ is the average time before that happens.
- **Dephasing:** the qubit keeps its 0/1 populations but loses the *relative phase* between them, the very ingredient that makes superposition and interference work. Its channel takes a "dephasing time" parameter: the timescale on which phase information randomizes away.
- **Readout error:** a flaky sensor. The qubit really was $|1\rangle$, but the measurement electronics report `0`, or vice versa.

This is why today's machines are called **NISQ** devices (Noisy Intermediate-Scale Quantum): mid-sized, and noisy enough that noise shapes what you can run on them.

To describe a noisy qubit we need one upgrade to the math. A perfectly known state is a **pure state**: a single vector $|\psi\rangle$. After noise strikes, the honest description becomes "with probability $p_1$ the state is $|\psi_1\rangle$, with probability $p_2$ it is $|\psi_2\rangle$, ...": a probability-weighted **mixture**, like a dict mapping states to probabilities. The object that captures this is the **density matrix**

$$\rho \;=\; \sum_k p_k\,|\psi_k\rangle\langle\psi_k|,$$

where, recalling Part 1's notation, $|\psi\rangle$ is a column vector and $\langle\psi|$ is its adjoint (`psi.conj().T`, a row vector), so each $|\psi_k\rangle\langle\psi_k|$ is a square matrix (column times row), and the sum weights those matrices by ordinary probabilities $p_k$. A pure state is the special case with a single term, $\rho = |\psi\rangle\langle\psi|$, which is exactly what Part 1's `as_density_matrix()` builds. Two facts to keep: the diagonal of $\rho$ holds the measurement probabilities of the basis states, so its **trace** ($\operatorname{Tr}$, the sum of the diagonal) is the total probability and always equals 1; and however you rotate the measurement basis, the probabilities you read off a valid $\rho$ never come out negative.

Here is the crucial subtlety, and the best one-line definition of decoherence. **A superposition is not a mixture.** The state $|+\rangle = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle)$ is ONE fully known state: you prepared it deterministically, and Part 2's H-then-H interference trick recovers $|0\rangle$ from it with certainty. A 50/50 *mixture* of $|0\rangle$ and $|1\rangle$ is genuine ignorance: `random.choice([ket0, ket1])`. Both answer a Z-basis measurement with a 50/50 coin flip, yet they are physically different states, and the difference lives in the **off-diagonal entries** of $\rho$. Noise is precisely the process that turns the first into the second. Watch:

*(One import note: this notebook mixes digital gates with analog Pauli observables, which share the names `X, Y, Z, I`. As in Parts 3 and 4, we alias the analog ones, `aX, aZ`, and so on.)*

In [ ]:
from qilisdk.core import ket, expect_val
from qilisdk.analog import X as aX

# |+>: a superposition, ONE fully known state
plus = (1 / np.sqrt(2)) * (ket(0) + ket(1))
rho_pure = plus.as_density_matrix()

# 50/50 classical mixture: genuine ignorance about which of |0>, |1> we hold
rho_mix = 0.5 * ket(0).as_density_matrix() + 0.5 * ket(1).as_density_matrix()

print("superposition rho:\n", np.round(rho_pure.dense().real, 3))
print("mixture rho:\n", np.round(rho_mix.dense().real, 3))

print("\nZ-basis probabilities, superposition:", rho_pure.probabilities())
print("Z-basis probabilities, mixture:      ", rho_mix.probabilities())

X_op = aX(0).to_qtensor()          # the Pauli-X observable as a matrix
print("\n<X> superposition:", expect_val(X_op, rho_pure).real)
print("<X> mixture:      ", expect_val(X_op, rho_mix).real)

Read the two matrices. The **diagonals are identical** (0.5 and 0.5: both states answer a computational-basis measurement with a fair coin), so Z-basis sampling can never tell them apart. The entire difference sits in the **off-diagonals**: 0.5 for the superposition, 0 for the mixture. Those off-diagonal entries are called **coherences**, and they are what interference runs on. Measuring along a different axis exposes them: $\langle X\rangle = 1.0$ for the superposition versus $0.0$ for the mixture.

Decoherence in one sentence: **noise eats the off-diagonals**, so your carefully engineered superposition silently degrades into a classical coin flip, and every interference-based trick from Parts 2 to 4 stops working. This is also why simulating noise requires density matrices: QiliSim switches to a density-matrix simulation automatically the moment you hand it a noise model.

## 5.1 · Noise channels & the `NoiseModel`

A **noise channel** is simply a function from a density matrix to a noisier density matrix. QiliSDK ships the standard ones as classes, and each has a one-line story a developer will recognize:

- **`BitFlip(probability=p)`**: with probability $p$, apply $X$ (swap $|0\rangle \leftrightarrow |1\rangle$). The network bit error of the quantum world.
- **`PhaseFlip(probability=p)`**: with probability $p$, apply $Z$ (flip the sign of the $|1\rangle$ branch). A quantum-only error with no classical cousin: invisible to plain sampling of a definite state, fatal to superposition. Demo below.
- **`Depolarizing(probability=p)`**: with probability $p$, apply one of $X$, $Y$, $Z$ chosen uniformly. "A bit of everything": the standard pessimistic model when all you know is a gate's error rate. (`PauliChannel` is the general version with a separate probability per Pauli.)
- **`AmplitudeDamping(t1=...)`** and **`Dephasing(t_phi=...)`**: the physical decay processes from 5.0, parameterized by device timescales (relaxation time, dephasing time) instead of a per-event probability.
- **`ReadoutAssignment(p01=..., p10=...)`**: the flaky sensor: classical misreads applied at measurement time.
- **`GaussianPerturbation` / `OffsetPerturbation`**: control noise: jitter or bias on gate *parameters*, a miscalibrated knob rather than a random error event.

> **Optional math aside (skippable).** Internally, digital noise channels are stored as **Kraus operators**: $\rho \mapsto \sum_k K_k\,\rho\,K_k^\dagger$, which in words says "one of a small set of error operations is applied, with the appropriate probability". Analog noise uses **Lindblad generators**, the continuous-time version of the same idea: a differential equation that mixes the state a little at every instant. One `NoiseModel` serves both worlds.

You build a `NoiseModel()`, `.add(...)` channels to it, and hand it to the backend. The **scope** of each channel is inferred from the keywords you pass:

- `nm.add(channel)` → **global**: applied to every qubit at every gate;
- `nm.add(channel, qubits=[0, 1])` → only those **qubits**;
- `nm.add(channel, gate=X)` → only after that **gate type** (pass both to target one gate type on specific qubits).

Here is the whole thesis in one comparison: an $X$ gate that *should* produce $|1\rangle$ every single time, given a 20 percent bit-flip error on the $X$ gate. Only the backend changes; the functional and readout are literally the same objects.

In [ ]:
from qilisdk.digital import Circuit, X
from qilisdk.functionals import DigitalPropagation
from qilisdk.readout import Readout
from qilisdk.backends import QiliSim
from qilisdk.noise import NoiseModel, BitFlip

circuit = Circuit(1)
circuit.add(X(0))                        # ideal outcome: |1> with certainty
functional = DigitalPropagation(circuit)
readout = Readout().with_sampling(nshots=2000)

print("IDEAL:", QiliSim().execute(functional, readout).get_samples())   # {'1': 2000}

nm = NoiseModel()
nm.add(BitFlip(probability=0.2), gate=X)   # 20% chance each X gate also flips the qubit
print("NOISY:", QiliSim(noise_model=nm).execute(functional, readout).get_samples())
# ~{'0': 400, '1': 1600}: your counts will scatter around the 20/80 split

### Phase errors: the quantum-only failure mode

A bit flip has an obvious classical cousin. A **phase flip** does not: it turns $\alpha|0\rangle + \beta|1\rangle$ into $\alpha|0\rangle - \beta|1\rangle$, and since measurement probabilities come from $|\alpha|^2$ and $|\beta|^2$ (the Born rule ignores signs), no measurement *of that state as it is* changes at all. So is it even an error? Compare two circuits:

1. **X, then measure.** The state is $|1\rangle$. A phase flip turns it into $-|1\rangle$, which is the same physical state: sampling sees nothing.
2. **H, then H.** Part 2's interference demo: the second $H$ undoes the first and $|0\rangle$ returns *with certainty*, but only because the two branches keep their relative phase. A phase flip between the two gates turns $|+\rangle$ into $|-\rangle$, and the second $H$ then lands on $|1\rangle$ instead of $|0\rangle$.

We use the exact tomography readout (tomography = reconstructing the full state, free on a simulator) so the numbers on screen are deterministic.

In [ ]:
from qilisdk.digital import H
from qilisdk.noise import PhaseFlip

exact = Readout().with_state_tomography()

nm_phase = NoiseModel()
nm_phase.add(PhaseFlip(probability=0.2))          # global: may strike after every gate

def exact_probs(circ, noise_model=None):
    result = QiliSim(noise_model=noise_model).execute(DigitalPropagation(circ), exact)
    return {k: round(v, 3) for k, v in result.state_tomography.probabilities.items()}

flip = Circuit(1)
flip.add(X(0))
print("X circuit,   ideal:     ", exact_probs(flip))
print("X circuit,   PhaseFlip: ", exact_probs(flip, nm_phase))

interfere = Circuit(1)
interfere.add(H(0))
interfere.add(H(0))
print("H-H circuit, ideal:     ", exact_probs(interfere))
print("H-H circuit, PhaseFlip: ", exact_probs(interfere, nm_phase))

The phase flip was completely invisible on the definite state, and it cost 20 percent of the answer in the interference circuit: $P(\text{wrong}) = 0.2$, exactly the flip probability. (Only a flip *between* the two $H$ gates matters: a flip after the second $H$ changes a sign but no probability.) The general lesson: **phase errors only show up once your algorithm relies on superposition and interference, which is exactly when quantum computing is useful.** A test suite that only checks definite bit values will never catch them, and that is one big reason quantum error correction is so much harder than classical error correction.

### 🧩 Exercise 5.1: noise on a Bell state

**Goal.** Take the Bell circuit from Part 2 ($H$ on qubit 0, `CNOT(0, 1)`, measure both), which ideally yields only `00` and `11`, and compare the sampled histograms **ideal vs noisy** under **depolarizing noise with probability 0.1 on both qubits**.

**Stakes.** Bell pairs are the raw resource for quantum-secure communication and for networking QPUs together. The weight your histogram leaks into the "forbidden" `01` and `10` outcomes is precisely why distributing entanglement over real channels is hard, and why labs obsess over Bell-pair quality. For reference: with this noise level the exact distribution puts probability 0.0333 on each forbidden outcome, so expect roughly 3 percent of your 2000 shots on each.

**Hints.**

- The scope rules in 5.1 show how to attach a channel to specific qubits.
- Ideal vs noisy means two `execute` calls that differ *only* in the backend object.
- The plotting block below activates once `noisy` holds a samples dict, and prints a reminder until then.

In [ ]:
from qilisdk.digital import CNOT, M
from qilisdk.noise import Depolarizing

bell = Circuit(2)
bell.add(H(0))
bell.add(CNOT(0, 1))
bell.add(M(0, 1))
bell_functional = DigitalPropagation(bell)
bell_readout = Readout().with_sampling(nshots=2000)

ideal = QiliSim().execute(bell_functional, bell_readout).get_samples()

# TODO: build a NoiseModel and add Depolarizing(probability=0.1) scoped to qubits 0 and 1,
#       then run the SAME bell_functional + bell_readout on a noisy backend and store the
#       resulting samples dict in `noisy`.
noisy = ...

if noisy is ...:
    print("Fill in `noisy` above, then rerun this cell to see the two histograms.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
    for ax, (title, counts) in zip(axes, [("ideal", ideal), ("noisy", noisy)]):
        keys = sorted(counts)
        ax.bar(keys, [counts[k] for k in keys])
        ax.set_title(title)
        ax.set_xlabel("bitstring")
    axes[0].set_ylabel("count over 2000 shots")
    plt.show()

## 5.2 · Real hardware numbers: why qubits forget

Time to plug in datasheet values. **$T_1$, the relaxation time,** is the single most quoted spec on every quantum vendor's status page: the average time an excited qubit survives before decaying to $|0\rangle$. For today's superconducting *transmon* qubits (the technology behind the chips of IBM, Google, and QiliSDK's maker Qilimanjaro), $T_1$ is tens to hundreds of microseconds. We will use a typical **50 microseconds**.

Physics predicts the decay exactly. Prepare $|1\rangle$, wait a time $t$, and the probability of still finding $|1\rangle$ is

$$P(1) = e^{-t/T_1},$$

the same exponential as a discharging capacitor or radioactive decay. The protocol behind that formula (prepare, wait, measure, repeat for longer and longer waits) is literally how labs measure $T_1$: it is a daily calibration ritual, not an exotic experiment. We now reproduce it with `AmplitudeDamping(t1=...)`. Two modeling choices to read closely:

- Each `I` (identity) gate in the circuit stands for **5 microseconds of deliberate waiting**. Real gates are much faster (tens to hundreds of *nano*seconds); we use long idles so the decay is visible within a handful of gates.
- Time-based channels need to know how long each gate takes. We set that with `NoiseConfig.set_default_gate_time(...)` and scope the channel with `gate=I`, so damping is applied only during the idles. **Sharp edge (verified on 0.2.0):** the per-gate-type `set_gate_time(gate, ...)` variant is *silently ignored* by QiliSim, so avoid it; the default time plus `gate=` scoping is the reliable pattern.

In [ ]:
from qilisdk.digital import I
from qilisdk.noise import NoiseConfig, AmplitudeDamping

T1 = 50e-6      # 50 microseconds: a typical transmon relaxation time
IDLE = 5e-6     # each I gate stands for 5 microseconds of waiting

config = NoiseConfig()
config.set_default_gate_time(IDLE)   # NOTE: per-gate set_gate_time is silently ignored in 0.2.0
nm_t1 = NoiseModel(config)
nm_t1.add(AmplitudeDamping(t1=T1), gate=I)   # relaxation acts only while idling

times_us = []
p1_simulated = []
print(" idles   wait (us)   P(1) simulated   exp(-t/T1)")
for n_idle in [0, 2, 4, 8, 12, 20]:
    c = Circuit(1)
    c.add(X(0))                       # prepare |1>
    for _ in range(n_idle):
        c.add(I(0))                   # wait 5 us
    res = QiliSim(noise_model=nm_t1).execute(DigitalPropagation(c), exact)
    p1 = res.state_tomography.probabilities["1"]
    t = n_idle * IDLE
    times_us.append(t * 1e6)
    p1_simulated.append(p1)
    print(f"{n_idle:>6}   {t * 1e6:>9.0f}   {p1:>14.4f}   {np.exp(-t / T1):>10.4f}")

t_grid = np.linspace(0, 100e-6, 200)
plt.plot(t_grid * 1e6, np.exp(-t_grid / T1), "--", label=r"theory $e^{-t/T_1}$")
plt.plot(times_us, p1_simulated, "o", label="simulated")
plt.xlabel("wait time (microseconds)")
plt.ylabel("P(still in |1>)")
plt.title("T1 relaxation of a transmon qubit, T1 = 50 us")
plt.legend()
plt.show()

The simulation matches $e^{-t/T_1}$ to four decimal places: after 10 microseconds $P(1) = 0.8187$, and after 100 microseconds only $0.1353$ of the signal survives. Now reread the table as an engineer: **circuit depth is a time budget.** Every gate spends wall-clock time on the chip, and the state you are computing with decays underneath you the entire while. A circuit whose runtime approaches a few times $T_1$ returns noise-flavored garbage no matter how clever the algorithm is. Keep this in mind for Part 6, where transpiling a logical circuit into hardware-native gates *multiplies* the gate count: that depth overhead is paid directly out of this budget.

## 5.3 · The payoff: how much noise can chemistry tolerate?

Time for this part's real deliverable. In Part 4 you computed the ground-state energy of the H$_2$ molecule with VQE and hit **chemical accuracy**: within **1.6 mHa** (milli-Hartree) of the exact answer, the bar a computed energy must clear before a chemist can trust predictions built on it. That ran on an ideal simulator. The question every quantum algorithm team must answer before renting QPU time is: **how noisy may the gates be before the result stops clearing the bar?** The analysis you are about to run, a noise budget for a target application, is a deliverable such teams actually produce.

The experiment takes two steps:

1. **Re-run the ideal VQE and freeze the winner.** We rebuild Part 4's setup with a *one-layer* hardware-efficient ansatz: one entangling layer already reaches the exact ground state of this 2-qubit molecule, and fewer gates means fewer places for noise to strike (we quantify the depth cost below). `result.optimal_parameters` is a plain list in `get_parameter_names()` order, so we zip the two into a dict and bind it with `set_parameters(...)`, turning the ansatz into a fixed, fully specified circuit.
2. **Re-evaluate the frozen circuit under increasing noise.** Same circuit, same exact-expectation readout (`nshots=0`), plus one *global* `Depolarizing` channel whose probability $p$ we sweep. Any change in the energy is purely the noise's doing.

Why expect the damage to grow linearly in $p$? Recall depolarizing noise: with probability $p$, one of $X$, $Y$, $Z$ strikes, chosen uniformly. Track a single $\langle Z\rangle$ expectation through one such event: with probability $1-p$ nothing happens and the value survives; a $Z$ error leaves $\langle Z\rangle$ untouched (contributing $+p/3$); $X$ and $Y$ errors flip its sign (contributing $-p/3$ each). Net: $\langle Z\rangle$ shrinks by the factor $(1-p) + p/3 - 2p/3 = 1 - 4p/3$ per noise event. (At $p = 3/4$ that factor hits zero: the state is fully mixed and measurement carries no information at all.) The molecular energy is a weighted sum of Pauli expectations like this one, so every noisy gate drains a proportional slice of the signal.

*(A note on step 1: COBYLA starts from random parameters, and on this one-layer problem it converged to the exact energy on every test run. If your printed VQE energy is not close to -1.8512 Ha, rerun the cell.)*

In [ ]:
from qilisdk.analog import I as aI, Y as aY, Z as aZ
from qilisdk.digital import CNOT, U3, HardwareEfficientAnsatz
from qilisdk.functionals import VariationalProgram
from qilisdk.optimizers import SciPyOptimizer
from qilisdk.cost_functions import ObservableCostFunction

# the H2 molecule at bond length 0.7414 angstrom (O'Malley et al., PRX 6, 031007 (2016))
g0, g1, g2, g3, g4, g5 = -0.4804, 0.3435, -0.4347, 0.5716, 0.0910, 0.0910
H_mol = (g0 * aI(0) + g1 * aZ(0) + g2 * aZ(1) + g3 * aZ(0) * aZ(1)
         + g4 * aY(0) * aY(1) + g5 * aX(0) * aX(1))
E_exact = np.linalg.eigvalsh(H_mol.to_matrix().toarray()).min()

# step 1: ideal VQE, then freeze the optimal parameters into the ansatz
ansatz = HardwareEfficientAnsatz(nqubits=2, layers=1, one_qubit_gate=U3, two_qubit_gate=CNOT)
program = VariationalProgram(
    functional=DigitalPropagation(ansatz),
    optimizer=SciPyOptimizer(method="COBYLA", tol=1e-9),
    cost_function=ObservableCostFunction(H_mol),
)
vqe_result = QiliSim().execute(program, Readout().with_state_tomography())

best = dict(zip(ansatz.get_parameter_names(), vqe_result.optimal_parameters))
ansatz.set_parameters(best)                  # the ansatz is now a fixed circuit
frozen = DigitalPropagation(ansatz)

print(f"exact ground energy: {E_exact:.6f} Ha")
print(f"ideal VQE energy:    {vqe_result.optimal_cost:.6f} Ha")

In [ ]:
from qilisdk.noise import Depolarizing
from qilisdk.core import expect_val

# step 2: re-evaluate the SAME frozen circuit under increasing gate noise.
# We read out the exact noisy state (a density matrix) and take <H> = Tr(H rho)
# ourselves with Part 1's expect_val. (The released 0.2.0 package raises a spurious
# "imaginary expectation value" error on the direct with_expectation path when noise
# is on and the observable has X or Y terms; reading the state sidesteps it.)
state_readout = Readout().with_state_tomography()
H_mol_op = H_mol.to_qtensor(2)                       # the observable as a padded matrix
CHEMICAL_ACCURACY = 1.6   # mHa

def h2_error_mha(p):
    # energy error (in mHa) of the frozen circuit under global depolarizing noise p
    if p == 0.0:
        backend = QiliSim()                          # ideal: no noise model at all
    else:
        nm = NoiseModel()
        nm.add(Depolarizing(probability=p))          # global: after every gate, every qubit
        backend = QiliSim(noise_model=nm)
    rho = backend.execute(frozen, state_readout).get_state()
    energy = float(complex(expect_val(H_mol_op, rho)).real)
    return (energy - E_exact) * 1000

ps = [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05]
errors = []
print("  p        error (mHa)   chemically accurate?")
for p in ps:
    err = h2_error_mha(p)
    errors.append(err)
    verdict = "yes" if err <= CHEMICAL_ACCURACY else "NO"
    print(f"{p:<8}   {err:>11.3f}   {verdict}")

plt.loglog(ps[1:], errors[1:], "o-", label="energy error")
plt.axhline(CHEMICAL_ACCURACY, color="tab:red", linestyle="--",
            label="chemical accuracy (1.6 mHa)")
plt.xlabel("depolarizing probability per gate, p")
plt.ylabel("energy error (mHa)")
plt.title("How much gate noise can the H2 result tolerate?")
plt.legend()
plt.show()

Read the table against where hardware actually stands. Today's best superconducting and trapped-ion devices quote two-qubit gate error rates between $10^{-3}$ and $10^{-4}$, in other words **99.9 to 99.99 percent gate fidelity** (fidelity here = $1 - p$, the probability a gate does its job cleanly). Our sweep says $p = 10^{-3}$ already misses chemical accuracy by more than a factor of two, while $p = 10^{-4}$ clears it with room to spare. **The difference between 99.9 and 99.99 percent fidelity is the difference between a useless and a useful chemistry result.** Current hardware sits exactly at that frontier, which is why every extra fidelity decimal makes headlines.

Two more observations worth pausing on:

- **Every noisy energy sits *above* the true ground energy** (all errors are positive). That is Part 4's variational principle at work: noise mixes higher-energy states into $\rho$, and mixing in anything can only raise the average. Handy in practice: you always know which way the noise biases you.
- **Deeper circuits degrade faster.** We verified the same sweep with Part 4's two-layer ansatz (double the gate count): the error at every $p$ roughly doubles. Depth costs signal: 5.2's time budget, now in expectation-value form.

### 🧩 Exercise 5.2: find the noise budget

**Goal.** The sweep brackets the failure point only loosely: $p = 10^{-4}$ passes (0.380 mHa) and $p = 10^{-3}$ fails (3.798 mHa). Tighten the bracket: probe $p = 0.0002$ and $p = 0.0005$ with the provided loop, decide between which two probes the budget lies, and state it the way a hardware roadmap would, as a **gate fidelity** ($1 - p$, in percent).

**Stakes.** "What error rate does application X need?" is one of the most consequential questions in the field: its answers steer hardware roadmaps and funding. You are computing a tiny but genuine instance of one.

**Hints.**

- `h2_error_mha(p)` from the sweep cell does the heavy lifting: one call per probe.
- Chemically accurate means the error is at most `CHEMICAL_ACCURACY` (1.6 mHa).
- Once one probe passes and the other fails, the budget lies between them: quoting it as a fidelity to two decimals (`99.9x %`) is enough.

In [ ]:
# Problem: bracket the largest depolarizing probability p that still keeps the H2 energy
# within chemical accuracy (1.6 mHa), then express the budget as a gate fidelity.

probes = []    # TODO: put your two probe probabilities here (between 0.0001 and 0.001)

for p in probes:
    err = h2_error_mha(p)
    verdict = "chemically accurate" if err <= CHEMICAL_ACCURACY else "too noisy"
    print(f"p = {p}: error {err:.3f} mHa -> {verdict}")

# TODO: from the two verdicts, state the noise budget as a gate fidelity in percent.

## Recap & what's next

- **Noise turns pure states into mixtures.** The density matrix $\rho$ describes both; the off-diagonal **coherences** are what decoherence destroys, and only interference-sensitive measurements (like $\langle X\rangle$ on $|+\rangle$) expose the loss.
- **Channels are functions on density matrices**, shipped as classes: `BitFlip`, `PhaseFlip`, `Depolarizing`, `AmplitudeDamping`, `Dephasing`, `ReadoutAssignment`, and the parameter perturbations. A `NoiseModel` collects them, with **scope** (global / `qubits=` / `gate=`) inferred from the `.add(...)` keywords.
- **Ideal vs noisy is one argument**: `QiliSim(noise_model=nm)`, functional and readout untouched. That is what makes a noise model a cheap staging environment for hardware runs.
- **Datasheet numbers reproduce lab physics**: `AmplitudeDamping(t1=50e-6)` matched $P(1) = e^{-t/T_1}$ to four decimals, and turned circuit depth into a concrete time budget.
- **Chemistry sets a hard noise budget**: the H$_2$ result survives only up to roughly $p \approx 4\times10^{-4}$ per gate (about 99.96 percent gate fidelity), squarely at the frontier of current hardware.
- Practical notes for 0.2.0: prefer the default deterministic density-matrix path (Monte Carlo trajectory averaging is unreliable in this release), and use `QiliSim` or `CudaBackend` for noisy runs (the QuTiP backend does not support noise).

**Next, Part 6, Execution & Toward Hardware:** the same programs retargeted across CPU, GPU, and QPU backends by one line; interop through OpenQASM and QIR; transpiling circuits to hardware-native gates (where this part's depth budget gets spent); and a quantum-reservoir capstone.